# 01 — Exploração dos dados (KDD, fases 1 a 3)

Notebook **guiado**: você completa os `TODO`s e responde às perguntas. A explicação do KDD está em `docs/guia/05-kdd.md`.

## Como usar

- **Rodar uma célula:** clique nela e aperte `Shift + Enter`. Rode de cima para baixo, sem pular.
- **`___` (três sublinhados)** marca o lugar que você completa. Se rodar sem completar, aparece `NameError: name '___' is not defined`. É normal, só quer dizer que falta preencher.
- **Dica:** cada `TODO` tem uma dica no comentário. Se travar, abra a "💡 Dica extra" logo abaixo, quando houver.
- **Verificação:** depois de cada `TODO` há uma célula que mostra ✅ (certo) ou ❌ (ainda não).
- **✍️ Perguntas:** responda com suas palavras, mesmo que seja um palpite. Errar o palpite e entender por quê é o objetivo.
- **Deu um erro que você não entende?** Copie a mensagem e pergunte ao Claude. Ler mensagens de erro faz parte do aprendizado.

> **Antes de começar:** o dataset precisa estar em `data/vangogh2photo/` (`python -m src.download_data`).

## 0. Preparação

Esta célula já está pronta. Ela importa as bibliotecas e cria a função `confere`, usada nas verificações.

In [ ]:
import sys
sys.path.insert(0, "..")  # permite importar o código da pasta src/

import torch
import matplotlib.pyplot as plt

from src.utils import DATASET_DIR, load_image, load_domain, to_float


def confere(condicao, mensagem):
    # Mostra ✅ se a condição for verdadeira, ❌ se ainda não.
    print(("✅ " if condicao else "❌ ") + mensagem)


def mostra(imagens, titulo=None, nrow=8):
    # Mostra um tensor uint8 (N, 3, H, W) como uma grade de imagens.
    from torchvision.utils import make_grid
    grade = make_grid(imagens, nrow=nrow, padding=2)
    plt.figure(figsize=(2 * nrow, 2 * ((len(imagens) + nrow - 1) // nrow)))
    plt.imshow(grade.permute(1, 2, 0))
    plt.axis("off")
    if titulo:
        plt.title(titulo)
    plt.show()


print("PyTorch:", torch.__version__)
print("GPU disponível:", torch.cuda.is_available())
print("Dataset em:", DATASET_DIR)

---
# Fase 1 do KDD: Seleção

Usamos o **`vangogh2photo`**, publicado junto com o artigo da CycleGAN. Ele tem dois **domínios**, cada um com uma pasta de treino e uma de teste:

| Pasta | O que tem |
|---|---|
| `trainA`, `testA` | pinturas de Van Gogh |
| `trainB`, `testB` | fotos de paisagens |

Os domínios **não têm pares**: nenhuma foto mostra a mesma cena de uma pintura. A CycleGAN aprende assim mesmo, e é por isso que ela foi escolhida (veja a ADR 0002).

### TODO 1: Quantas imagens há em cada pasta?

In [ ]:
arquivos = {pasta: sorted((DATASET_DIR / pasta).glob("*.jpg"))
            for pasta in ("trainA", "testA", "trainB", "testB")}

# TODO 1: conte quantos arquivos há na lista de cada pasta.
# Dica: arquivos["trainA"] é uma lista; len(lista) dá o tamanho dela.
n_pinturas_treino = ___
n_fotos_treino = ___

print("Pinturas (treino):", n_pinturas_treino, "| teste:", len(arquivos["testA"]))
print("Fotos (treino):", n_fotos_treino, "| teste:", len(arquivos["testB"]))
print(f"Há {n_fotos_treino / n_pinturas_treino:.1f} fotos para cada pintura")

In [ ]:
confere(n_pinturas_treino == 400, "pinturas de treino")
confere(n_fotos_treino == 6287, "fotos de treino")

**Perguntas:**
1. Os dois domínios estão equilibrados? Quantas vezes mais fotos do que pinturas?
2. O discriminador de pinturas (D_pintura) só vê essas poucas pinturas reais o treino inteiro. Que risco isso traz?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

### TODO 2: Uma imagem vista como números

Uma imagem colorida é um **tensor** com 3 dimensões: canais x altura x largura. Os 3 canais são vermelho, verde e azul (RGB).

In [ ]:
# TODO 2: escolha um número entre 0 e n_pinturas_treino - 1 para ver aquela pintura.
i = ___

img = load_image(arquivos["trainA"][i])
print("Formato:", tuple(img.shape), "| tipo:", img.dtype)

plt.imshow(img.permute(1, 2, 0))   # o matplotlib quer (altura, largura, canais)
plt.title(arquivos["trainA"][i].name)
plt.axis("off")
plt.show()

In [ ]:
confere(img.shape[0] == 3, "3 canais de cor")

Agora os números de um pedaço de 4x4 pixels no canto da imagem, separados por canal:

In [ ]:
for c, nome in enumerate(["vermelho", "verde", "azul"]):
    print(nome)
    print(img[c, :4, :4])

**Perguntas:**
1. O formato tem três números. O que cada um significa?
2. Um pixel com os valores (255, 255, 0) nos três canais: que cor ele tem?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

Veja algumas imagens de cada domínio (célula pronta):

In [ ]:
mostra(torch.stack([load_image(p, 128) for p in arquivos["trainA"][:8]]), "pinturas de Van Gogh")
mostra(torch.stack([load_image(p, 128) for p in arquivos["trainB"][:8]]), "fotos")

---
# Fase 2 do KDD: Pré-processamento (qualidade dos dados)

Antes de modelar, conferimos se os dados têm problemas: tamanhos diferentes, imagens repetidas, imagens que não deveriam estar ali.

### TODO 3: Todas as imagens têm o mesmo tamanho?

In [ ]:
from collections import Counter
from PIL import Image

tamanhos = Counter()
for pasta, lista in arquivos.items():
    for p in lista:
        with Image.open(p) as im:
            tamanhos[(pasta, im.size, im.mode)] += 1

for (pasta, tamanho, modo), n in sorted(tamanhos.items()):
    print(f"{pasta:<7} {tamanho} {modo}: {n}")

# TODO 3: quantas combinações diferentes de (pasta, tamanho, modo) apareceram?
# Dica: tamanhos é um dicionário; len() também funciona com dicionários.
n_combinacoes = ___

In [ ]:
confere(n_combinacoes == 4, "4 combinações: uma por pasta, ou seja, todas as imagens de cada pasta são iguais em tamanho e modo")

**Pergunta:** o modo `RGB` indica 3 canais. Se aparecesse uma imagem em `L` (escala de cinza), o que daria errado no treino? (Dica: olhe `load_image` em `src/utils.py`, que usa `.convert("RGB")`.)

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

### TODO 4: As pinturas de teste são diferentes das de treino?

Para avaliar o modelo de forma justa, as imagens de teste não podem estar no treino. Vamos carregar todas as imagens em 128x128 (a resolução do treino) e procurar, para cada pintura de teste, a pintura de treino **mais parecida**, medindo a **distância L2**: quanto menor, mais parecidas. Distância 0 = imagens idênticas.

Na primeira vez, `load_domain` lê e redimensiona todas as imagens e salva o resultado em `data/vangogh2photo_128/`. Pode levar um minuto.

In [ ]:
pinturas_treino = load_domain("pintura", train=True)
pinturas_teste = load_domain("pintura", train=False)
fotos_treino = load_domain("foto", train=True)
fotos_teste = load_domain("foto", train=False)

# compute_mode: faz a conta da distância do jeito exato (explicação logo abaixo)
distancias = torch.cdist(pinturas_teste.flatten(1).float(), pinturas_treino.flatten(1).float(),
                         compute_mode="donot_use_mm_for_euclid_dist")
dist_min, idx_vizinho = distancias.min(dim=1)   # a menor distância de cada pintura de teste

# TODO 4: quantas pinturas de teste têm uma cópia exata (distância < 1) no treino?
# Dica: (dist_min < 1) dá um tensor de True/False; .sum() conta os True.
n_copias = ___
print(f"Pinturas de teste com cópia no treino: {n_copias} de {len(pinturas_teste)}")

In [ ]:
confere(n_copias == (dist_min < 1).sum().item(), "contagem de cópias")

Veja 5 pares lado a lado (célula pronta):

In [ ]:
mostra(torch.cat([pinturas_teste[:5], pinturas_treino[idx_vizinho[:5]]]), "teste (em cima) x treino mais parecida (embaixo)", nrow=5)

**Perguntas:**
1. Quantas pinturas de teste são cópias de pinturas de treino? O que isso significa para a ideia de "testar com imagens que o modelo nunca viu"?
2. Na Etapa 4, queremos saber se a GAN **copia** pinturas. Para isso precisamos de uma referência: o quanto duas pinturas **diferentes** de Van Gogh costumam se parecer. Se o teste não serve para isso, que outra comparação você faria usando só as 400 pinturas? (Veja como o `src/evaluate.py` resolveu, depois de pensar.)

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

<details><summary>💡 Por que o compute_mode?</summary>

Por padrão, o `torch.cdist` calcula a distância com um atalho: `|a - b|² = |a|² + |b|² - 2·a·b`, que vira uma multiplicação de matrizes e é muito mais rápido. Mas aqui cada imagem tem 49.152 números entre 0 e 255, então `|a|²` e `|b|²` são enormes. Quando `a` e `b` são quase iguais, a conta subtrai dois números enormes e quase iguais, e o erro de arredondamento do computador fica maior que a própria diferença. Com o atalho, este mesmo código dizia que só 222 das 400 pinturas eram cópias, e as outras pareciam ter distâncias entre 10 e 25. O `compute_mode="donot_use_mm_for_euclid_dist"` faz a conta direta, `(a - b)` elemento por elemento, e mostra a verdade.

Lição: quando um número parece estranho, desconfie também da conta, não só dos dados.
</details>

### TODO 5: As cores das pinturas e das fotos

O gerador precisa transformar as cores de uma foto nas cores de Van Gogh. Quanto elas diferem?

In [ ]:
# TODO 5: calcule a média de cada canal (vermelho, verde, azul) nas pinturas de treino.
# Dica: o tensor tem formato (N, 3, 128, 128). A média deve "sumir" com as dimensões 0, 2 e 3,
#       sobrando só a dos canais: tensor.float().mean(dim=(0, 2, 3))
media_pinturas = ___
media_fotos = fotos_treino.float().mean(dim=(0, 2, 3))

print("Média RGB pinturas:", media_pinturas.round().tolist())
print("Média RGB fotos:   ", media_fotos.round().tolist())

fig, eixos = plt.subplots(1, 3, figsize=(14, 3))
for c, (nome, cor) in enumerate([("vermelho", "r"), ("verde", "g"), ("azul", "b")]):
    eixos[c].hist(pinturas_treino[:, c].flatten()[::50], bins=50, alpha=0.6, density=True, color=cor, label="pinturas")
    eixos[c].hist(fotos_treino[:, c].flatten()[::50], bins=50, alpha=0.4, density=True, color="gray", label="fotos")
    eixos[c].set_title(nome)
    eixos[c].legend()
plt.show()

In [ ]:
confere(tuple(media_pinturas.shape) == (3,), "uma média por canal")

**Perguntas:**
1. Em qual canal pinturas e fotos mais diferem?
2. As fotos têm mais pixels muito escuros (perto de 0) ou muito claros (perto de 255) do que as pinturas? O que isso diz sobre o contraste de cada domínio?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

---
# Fase 3 do KDD: Transformação

Redes neurais treinam melhor com números pequenos e centrados. O gerador termina com a função `tanh`, que só produz valores entre **-1 e 1** (veja `src/models.py`). Por isso as imagens reais precisam estar na mesma faixa.

### TODO 6: Converter de [0, 255] para [-1, 1]

In [ ]:
# TODO 6: transforme pinturas_treino para a faixa [-1, 1].
# Dica 1: primeiro converta para número decimal com .float()
# Dica 2: dividir por 127.5 leva [0, 255] para [0, 2]. O que falta para chegar em [-1, 1]?
x_norm = ___

print("Menor:", x_norm.min().item(), "| Maior:", x_norm.max().item())

<details><summary>💡 Dica extra</summary>

Faça a conta com os extremos: 0 / 127.5 = 0 e 255 / 127.5 = 2. Que operação transforma 0 em -1 e 2 em 1?
</details>

In [ ]:
confere(abs(x_norm.min().item() + 1) < 1e-6 and abs(x_norm.max().item() - 1) < 1e-6, "faixa [-1, 1]")
confere(torch.allclose(x_norm, to_float(pinturas_treino)), "igual à transformação usada no projeto (to_float)")

### TODO 7: Espelhar as imagens (*data augmentation*)

Com poucas pinturas, um truque é **espelhar** as imagens na horizontal durante o treino: uma pintura espelhada continua parecendo um Van Gogh, e o modelo vê "mais" exemplos. A CycleGAN faz isso (veja `random_batch` em `src/utils.py`).

In [ ]:
pintura = pinturas_treino[3]

# TODO 7: espelhe a pintura na horizontal (inverta a ordem das colunas).
# Dica: tensor.flip(dim) inverte a ordem na dimensão dim. A largura é a última dimensão: -1.
espelhada = ___
de_cabeca_para_baixo = pintura.flip(-2)

mostra(torch.stack([pintura, espelhada, de_cabeca_para_baixo]), "original | espelhada | de cabeça para baixo", nrow=3)

In [ ]:
confere(torch.equal(espelhada[:, :, 0], pintura[:, :, -1]), "a primeira coluna virou a última")

**Pergunta:** o treino espelha as imagens na horizontal, mas **nunca** na vertical. Pensando no objetivo do projeto (pintar o céu), por que virar de cabeça para baixo seria uma má ideia?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

---
# Explorando padrões

### TODO 8: A "imagem média" de cada domínio

Tirando a média pixel a pixel de todas as imagens, os detalhes somem e sobra a **composição típica**: onde costuma ficar o claro, o escuro, cada cor.

In [ ]:
# TODO 8: calcule a imagem média das fotos de treino.
# Dica: a média deve "sumir" só com a dimensão 0 (as imagens): .float().mean(dim=0)
media_img_fotos = ___
media_img_pinturas = pinturas_treino.float().mean(dim=0)

mostra(torch.stack([media_img_pinturas, media_img_fotos]).round().byte(), "pintura média | foto média", nrow=2)

In [ ]:
confere(tuple(media_img_fotos.shape) == (3, 128, 128), "uma imagem média de 3x128x128")

**Perguntas:**
1. Na foto média, o que aparece na parte de cima? E na de baixo?
2. Isso é uma boa ou má notícia para o projeto, que quer pintar justamente o céu?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

### TODO 9: Projeção 2D com PCA

Cada imagem 128x128 colorida tem 49.152 números, impossível de desenhar. O **PCA** (Análise de Componentes Principais) é uma técnica de **aprendizado não supervisionado**: ela encontra as direções em que os dados mais variam e "achata" as imagens em 2 números, perdendo o mínimo de informação.

Para ficar rápido, as imagens são reduzidas para 32x32 antes. O PCA **não sabe** quais são pinturas e quais são fotos: as cores no gráfico são só para conferirmos.

In [ ]:
import numpy as np
import torch.nn.functional as nnf
from sklearn.decomposition import PCA

n = 400
pequenas = nnf.interpolate(torch.cat([pinturas_treino[:n], fotos_treino[:n]]).float() / 255, size=32, mode="area")
X = pequenas.flatten(1).numpy()           # 800 imagens, cada uma com 3 x 32 x 32 = 3072 números
dominio = np.array([0] * n + [1] * n)     # 0 = pintura, 1 = foto

# TODO 9: crie um PCA com 2 componentes e aplique em X.
# Dica: pca = PCA(n_components=...)   e depois   X_2d = pca.fit_transform(...)
pca = ___
X_2d = ___

plt.figure(figsize=(7, 6))
for d, nome in enumerate(["pinturas", "fotos"]):
    plt.scatter(X_2d[dominio == d, 0], X_2d[dominio == d, 1], s=6, alpha=0.6, label=nome)
plt.legend()
plt.title("Pinturas e fotos projetadas em 2D (PCA)")
plt.show()
print(f"Variância explicada pelos 2 componentes: {pca.explained_variance_ratio_.sum():.1%}")

In [ ]:
confere(X_2d.shape == (800, 2), "projeção em 2D")

**Perguntas:**
1. Pinturas e fotos formam grupos separados ou se misturam?
2. **Previsão para a Etapa 4:** depois do treino, as fotos convertidas pela GAN vão ser projetadas neste mesmo gráfico. Para que lado você espera que elas caiam?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

### 🏆 Desafio opcional: t-SNE

O **t-SNE** é outra técnica de projeção, não linear, que costuma separar melhor os grupos. Tente escrever sozinho, adaptando o código do PCA:

- importe com `from sklearn.manifold import TSNE`;
- use `TSNE(n_components=2, random_state=42).fit_transform(X)`.

In [ ]:
# Desafio: seu código do t-SNE aqui

---
# Contexto dos dados (para a análise ética)

As pinturas foram coletadas do **WikiArt** e as fotos, do **Flickr**, pelos autores da CycleGAN. Pesquise rapidamente e responda:

1. Quem escolheu quais pinturas de Van Gogh entraram? Que fases da obra dele podem estar sub-representadas?
2. As fotos são paisagens. Que tipos de foto (noite, cidade, interiores, pessoas) o modelo provavelmente nunca viu?
3. Van Gogh é domínio público. O que mudaria, eticamente, se este dataset fosse de um artista vivo?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

---
# Resumo: o que descobri

Preencha a tabela. Depois, peça ao Claude para registrar isso em `docs/experimentos/`, porque essas conclusões vão para o relatório.

| Pergunta | O que descobri |
|---|---|
| Tamanho da base (pinturas / fotos, treino e teste) | |
| Proporção fotos por pintura | |
| Tamanho e formato das imagens | |
| Pinturas de teste repetidas no treino? | |
| Maior diferença de cor entre os domínios | |
| Composição típica das fotos (foto média) | |
| Pinturas e fotos se separam no PCA? | |
| Possíveis vieses da base | |